In [4]:
# ============================================================
# 07_R_gpu_anvl_post_estimation.ipynb
# ANVL GPU post-estimation benchmark for the Apollo Swiss
# route-choice MXL model
#
# This notebook benchmarks GPU-based estimation and
# post-estimation for the panel mixed-logit specification used
# in the Apollo Swiss route-choice example. ANVL provides the
# CUDA likelihood and analytical score evaluator, while BGW is
# used for maximum-likelihood estimation.
#
# Scope:
# - GPU execution through ANVL with the PJRT CUDA backend.
# - Deterministic Halton draws for four lognormal coefficients.
# - BGW estimation using individual likelihoods and analytical scores.
# - Post-estimation through postestimation_bgw().
# - Draw sizes of 500, 1,000, 2,000, 5,000, and 10,000.
# - Separate timing of JIT compilation, likelihood evaluation,
#   estimation, and post-estimation.
# - Post-estimation coefficient tables reported by draw size.
# - GPU, package, and R session records for reproducibility.
#
# Main outputs:
# - results/benchmark_summary_anvl_bgw_gpu.xlsx
# - results/session_info_anvl_gpu.txt
# - Printed post-estimation coefficient tables for each draw size.
# ============================================================

In [5]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Packages are installed only when missing. The benchmark depends on
# BGW for optimisation, and randtoolbox for Halton draws.

apt_pkgs <- c(
  "build-essential", "cmake", "pkg-config",
  "libprotobuf-dev", "libprotoc-dev", "protobuf-compiler",
  "libuv1-dev", "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev"
)

stopifnot(system2("apt-get", c("update", "-qq")) == 0L)
stopifnot(system2("apt-get", c("install", "-y", apt_pkgs)) == 0L)

# 2. R packages

Sys.setenv(
  CUDA_COMPONENTS = "nvshmem,nccl,nvcc",
  PJRT_INSTALL    = "1"
)

options(repos = c(
  "https://r-xla.r-universe.dev",
  "https://mlverse.r-universe.dev",
  "https://cloud.r-project.org"
))

pkgs <- c(
  "randtoolbox", "bgw", "maxLik",
  "cuda12.8", "pjrt", "anvl"
)

for (pkg in pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

# 3. Load packages

suppressPackageStartupMessages({
  library(randtoolbox)
  library(bgw)
  library(maxLik)
  library(pjrt)
  library(anvl)
})


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rngWELL’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zoo’, ‘miscTools’, ‘sandwich’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘checkmate’, ‘safetensors’, ‘tengen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘stablehlo’, ‘xlamisc’




In [6]:
# ============================================================
# Fast Apollo installation in Colab using Posit Package Manager
# ============================================================

options(HTTPUserAgent = sprintf(
  "R/%s R (%s)",
  getRversion(),
  paste(
    getRversion(),
    R.version["platform"],
    R.version["arch"],
    R.version["os"]
  )
))

# Auto-detect Ubuntu codename, e.g., jammy or noble
ubuntu_codename <- tryCatch(
  system("bash -lc '. /etc/os-release; echo ${VERSION_CODENAME}'", intern = TRUE),
  error = function(e) "jammy"
)

ppm_repo <- sprintf(
  "https://packagemanager.posit.co/cran/__linux__/%s/latest",
  ubuntu_codename[1]
)

options(repos = c(CRAN = ppm_repo))

cat("Using CRAN repo:\n", getOption("repos"), "\n\n")

install.packages("apollo", Ncpus = parallel::detectCores())

library(apollo)

pkg_versions <- data.frame(
  package = c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl"),
  version = vapply(
    c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl"),
    function(pkg) as.character(packageVersion(pkg)),
    character(1)
  )
)

print(pkg_versions)

Using CRAN repo:
 https://packagemanager.posit.co/cran/__linux__/jammy/latest 



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘SparseM’, ‘MatrixModels’, ‘truncnorm’, ‘future.apply’, ‘mcmc’, ‘quantreg’, ‘mnormt’, ‘mvtnorm’, ‘numDeriv’, ‘Deriv’, ‘matrixStats’, ‘coda’, ‘Rsolnp’, ‘MCMCpack’, ‘RcppArmadillo’, ‘RcppEigen’




             . ,,                                                            
            ,      ,,                                                        
 ,,,,,,    ,         ,,                                                      
,     ,,  ,            ,,,,.                                                 
,,     , ,,   ,,,,,,    ,,,                                 //  //           
  ,     ,,,.   ,,,,,.   ,,      ////                        //  //           
,,     ,,,,,.           ,,     // //     //////    /////    //  //    /////  
,,,        ,,           ,      //  //    /    //  //   //   //  //   //   // 
              ,,  

                package version
apollo           apollo   0.3.9
randtoolbox randtoolbox   2.0.5
bgw                 bgw   0.1.4
maxLik           maxLik 1.5.2.2
cuda12.8       cuda12.8   1.0.9
pjrt               pjrt   0.5.0
anvl               anvl   0.4.0


In [7]:
# -------------------------------------------------------------------
# 2. Project paths
# -------------------------------------------------------------------

proj_root <- getwd()
raw_results_dir <- file.path(proj_root, "results", "raw")
summary_file <- file.path(proj_root, "results", "benchmark_summary_anvl_bgw_gpu.xlsx")
session_info_file <- file.path(proj_root, "results", "session_info_anvl_gpu.txt")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)

benchmark_results <- list()


In [8]:
# ============================================================
# 3. Define benchmark settings
# ============================================================

# This section defines common settings and input builders for
# ANVL + maxLik + GPU and ANVL + BGW + GPU.

# ------------------------------------------------------------
# 3.1 Benchmark settings
# ------------------------------------------------------------

seed <- 123L
# Standard deterministic Halton draws; seed is not used.
device_use <- "cuda"

# ------------------------------------------------------------
# 3.2 Load data and build ANVL database
# ------------------------------------------------------------
# Create ANVL data arrays with dimension task × individual × draw.

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL

n_indiv <- length(unique(database$ID))
n_task <- as.integer(nrow(database) / n_indiv)
n_obs <- nrow(database)

make_database_nv <- function() {

  make_x <- function(x) {
    nv_array(
      array(
        as.numeric(x),
        dim = c(n_task, n_indiv, 1L)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt1 = make_x(database$tt1),
    tc1 = make_x(database$tc1),
    hw1 = make_x(database$hw1),
    ch1 = make_x(database$ch1),

    tt2 = make_x(database$tt2),
    tc2 = make_x(database$tc2),
    hw2 = make_x(database$hw2),
    ch2 = make_x(database$ch2),

    choice1 = make_x(database$choice == 1),
    choice2 = make_x(database$choice == 2)
  )
}

# ------------------------------------------------------------
# 3.3 Generate Halton normal draws
# ------------------------------------------------------------
# Create ANVL draw arrays with dimension task × individual × draw.

make_draws_nv <- function(num_draws) {

  halton_sequence <- halton(
    n   = n_indiv * num_draws,
    dim = 4
  )

  norm_value <- qnorm(halton_sequence)

  make_draw <- function(z) {

    draw_ind <- matrix(
      z,
      nrow = n_indiv,
      ncol = num_draws,
      byrow = TRUE
    )

    nv_array(
      array(
        as.numeric(draw_ind),
        dim = c(1L, n_indiv, num_draws)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt = make_draw(norm_value[, 1]),
    tc = make_draw(norm_value[, 2]),
    hw = make_draw(norm_value[, 3]),
    ch = make_draw(norm_value[, 4])
  )
}

# ------------------------------------------------------------
# 3.4 Print benchmark information
# ------------------------------------------------------------

cat("Benchmark settings\n")
cat("Seed         :", seed, "\n")
cat("Device       :", device_use, "\n")
cat("Individuals  :", n_indiv, "\n")
cat("Tasks/person :", n_task, "\n")
cat("Observations :", n_obs, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

# ------------------------------------------------------------
# 3.5 Verify GPU backend and save environment information
# ------------------------------------------------------------

cat("\nANVL device check:\n")
print(nv_array(1, dtype = "f64", device = device_use))

nvidia_smi <- tryCatch(
  system("nvidia-smi", intern = TRUE),
  error = function(e) paste("nvidia-smi failed:", conditionMessage(e))
)

cat("\nNVIDIA GPU check:\n")
cat(paste(nvidia_smi, collapse = "\n"), "\n")

sink(session_info_file)
cat("R session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
for (pkg in c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl")) {
  cat(pkg, ": ", as.character(packageVersion(pkg)), "\n", sep = "")
}

cat("\nANVL device check\n")
print(nv_array(1, dtype = "f64", device = device_use))

cat("\nNVIDIA GPU information\n")
cat(paste(nvidia_smi, collapse = "\n"), "\n")
sink()

cat("Session and hardware information saved to:", session_info_file, "\n")


Benchmark settings
Seed         : 123 
Device       : cuda 
Individuals  : 388 
Tasks/person : 9 
Observations : 3492 
Raw output   : /content/results/raw 
Summary file : /content/results/benchmark_summary_anvl_bgw_gpu.xlsx 

ANVL device check:


<https://github.com/zml/pjrt-artifacts/releases/download/v14.0.1/pjrt-cuda_linux-amd64.tar.gz>


AnvlArray
 1
[ CUDAf64{1} ] 

NVIDIA GPU check:
Mon Aug 17 22:46:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             27W /   70W |   11299MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |


In [ ]:
# ============================================================
# 4. Define ANVL + BGW likelihood function
# ============================================================

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [10]:
# ============================================================
# 5. BGW estimation
# ============================================================
# Run BGW with individual likelihoods and analytical scores.

estimate_bgw <- function(calcR, calcJ, start_value) {
  bgw::bgw_mle(
    calcR = calcR,
    calcJ = calcJ,
    betaStart = start_value,
    bgw_settings = list(vcHessianMethod = "bhhh")
  )
}

In [ ]:
# ============================================================
# 6. BGW post-estimation
# ============================================================
# Recompute the Hessian from the summed analytical score and
# form conventional and sandwich-robust covariance matrices.

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [12]:
# ============================================================
# 7.1 500-draw
# ============================================================

num_draws <- 500L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253777958e+03
     1     2 1.775420248e+03 2.122e-01 9.248e-03 1.97e-03   G   2.26e+01 9.30e-01 5.390e-02
     2     3 1.764574965e+03 6.109e-03 3.854e-02 8.47e-02   G   0.00e+00 2.65e+01 3.854e-02
     3     4 1.569217403e+03 1.107e-01 4.734e-02 1.74e-01   G   0.00e+00 2.43e+01 4.734e-02
     4     5 1.503827816e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9961749,0.09022569,-22.124241,0.12227515,-16.325270
sigma_log_b_tt,0.4800606,0.07174704,6.691016,0.06593527,7.280787
mu_log_b_tc,-1.0297974,0.15127314,-6.807537,0.21132500,-4.873051
sigma_log_b_tc,-0.9498845,0.10571090,-8.985681,0.13088452,-7.257424
mu_log_b_hw,-2.9450129,0.08497437,-34.657663,0.09104751,-32.345893
sigma_log_b_hw,0.8180315,0.11321679,7.225355,0.12724778,6.428651
mu_log_b_ch,0.6270705,0.07579221,8.273548,0.08650655,7.248821
sigma_log_b_ch,-0.8480168,0.12900722,-6.573406,0.17670890,-4.798948


In [13]:
# ============================================================
# 7.2 1000-draw
# ============================================================

num_draws <- 1000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253776970e+03
     1     2 1.782933462e+03 2.089e-01 9.552e-03 1.89e-03   G   2.35e+01 9.26e-01 5.409e-02
     2     3 1.739550911e+03 2.433e-02 3.590e-02 9.08e-02   G   0.00e+00 2.69e+01 3.590e-02
     3     5 1.537309611e+03 1.163e-01 4.772e-02 1.58e-01  S-G  0.00e+00 2.38e+01 4.772e-02
     4     6 1.475331487e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9987308,0.08826221,-22.645373,0.11330802,-17.639800
sigma_log_b_tt,0.4732843,0.08251688,5.735606,0.08334580,5.678562
mu_log_b_tc,-1.0542865,0.13158661,-8.012111,0.16151515,-6.527477
sigma_log_b_tc,-1.0006007,0.08070687,-12.397963,0.07234435,-13.831083
mu_log_b_hw,-2.9374570,0.08286230,-35.449859,0.08958839,-32.788365
sigma_log_b_hw,0.7746465,0.10475476,7.394857,0.11043228,7.014674
mu_log_b_ch,0.6293025,0.07513538,8.375581,0.08353806,7.533123
sigma_log_b_ch,-0.8555308,0.11435071,-7.481640,0.13405498,-6.381940


In [14]:
# ============================================================
# 7.3 2000-draw
# ============================================================

num_draws <- 2000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253780344e+03
     1     2 1.785623676e+03 2.077e-01 9.730e-03 1.85e-03   G   2.40e+01 9.26e-01 5.445e-02
     2     3 1.747677032e+03 2.125e-02 3.339e-02 9.40e-02   G   0.00e+00 2.77e+01 3.339e-02
     3     5 1.534470233e+03 1.220e-01 5.094e-02 1.78e-01  S-G  0.00e+00 2.51e+01 5.094e-02
     4     6 1.471348289e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9888888,0.08810972,-22.572863,0.11423135,-17.411058
sigma_log_b_tt,0.4679015,0.07140122,6.553130,0.06297130,7.430394
mu_log_b_tc,-1.0299893,0.13406829,-7.682573,0.16842840,-6.115295
sigma_log_b_tc,-0.9918300,0.08436408,-11.756544,0.07682109,-12.910909
mu_log_b_hw,-2.9364141,0.08599252,-34.147320,0.09092715,-32.294139
sigma_log_b_hw,0.8471772,0.10851380,7.807092,0.10919094,7.758677
mu_log_b_ch,0.6299280,0.07398230,8.514577,0.08216912,7.666238
sigma_log_b_ch,-0.8360589,0.10473687,-7.982470,0.11554672,-7.235679


In [15]:
# ============================================================
# 7.4 5000-draw
# ============================================================

num_draws <- 5000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253779674e+03
     1     2 1.785870019e+03 2.076e-01 9.750e-03 1.84e-03   G   2.40e+01 9.26e-01 5.503e-02
     2     3 1.711901489e+03 4.142e-02 3.539e-02 9.09e-02   G   0.00e+00 2.75e+01 3.539e-02
     3     4 1.588050492e+03 7.235e-02 4.585e-02 1.75e-01   G   0.00e+00 2.53e+01 4.585e-02
     4     5 1.559316211e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9960589,0.08891350,-22.449448,0.11434259,-17.456828
sigma_log_b_tt,0.4782521,0.07790631,6.138812,0.07254816,6.592202
mu_log_b_tc,-1.0348794,0.13722430,-7.541517,0.17262539,-5.994943
sigma_log_b_tc,-1.0020086,0.09642899,-10.391155,0.09616045,-10.420173
mu_log_b_hw,-2.9359168,0.08475454,-34.640230,0.09093118,-32.287241
sigma_log_b_hw,0.8127984,0.11493492,7.071814,0.12709796,6.395055
mu_log_b_ch,0.6271475,0.07369832,8.509657,0.08186439,7.660809
sigma_log_b_ch,-0.8308798,0.10277890,-8.084148,0.10975866,-7.570061


In [16]:
# ============================================================
# 7.5 10000-draw
# ============================================================

num_draws <- 10000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253780344e+03
     1     2 1.785651884e+03 2.077e-01 9.757e-03 1.84e-03   G   2.41e+01 9.26e-01 5.580e-02
     2     3 1.721569344e+03 3.589e-02 3.443e-02 9.67e-02   G   0.00e+00 2.84e+01 3.443e-02
     3     4 1.529585265e+03 1.115e-01 4.909e-02 1.77e-01   G   0.00e+00 2.62e+01 4.909e-02
     4     6 1.498819994e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9951397,0.08879316,-22.469521,0.11473770,-17.388703
sigma_log_b_tt,0.4744672,0.07323035,6.479106,0.06537667,7.257440
mu_log_b_tc,-1.0358588,0.13786167,-7.513755,0.17459302,-5.932991
sigma_log_b_tc,-1.0008916,0.09441622,-10.600845,0.09477361,-10.560868
mu_log_b_hw,-2.9349687,0.08500399,-34.527423,0.09124124,-32.167130
sigma_log_b_hw,0.8157620,0.11716682,6.962398,0.13023658,6.263693
mu_log_b_ch,0.6291527,0.07406281,8.494852,0.08200609,7.672024
sigma_log_b_ch,-0.8418874,0.10250432,-8.213190,0.10765191,-7.820460


In [17]:
# ============================================================
# 8. Summarise and export results
# ============================================================

if (!length(benchmark_results)) {
  stop("Run at least one Section 7 cell before exporting results.")
}

benchmark_timings <- do.call(rbind, benchmark_results)
row.names(benchmark_timings) <- NULL

gpu_name <- trimws(system(
  "nvidia-smi --query-gpu=name --format=csv,noheader",
  intern = TRUE
)[1L])

benchmark_summary <- data.frame(
  package = "ANVL",
  implementation = "anvl_bgw_gpu",
  backend = "GPU",
  device = gpu_name,
  optimiser = "BGW",
  draws = benchmark_timings$draws,
  seed = seed,
  jit_time_sec = benchmark_timings$jit_time_sec,
  likelihood_eval_time_sec = benchmark_timings$likelihood_eval_time_sec,
  initial_loglik = benchmark_timings$initial_loglik,
  final_loglik = benchmark_timings$final_loglik,
  mean_estimation_time_sec = benchmark_timings$estimation_time_sec,
  postestimation_time_sec = benchmark_timings$postestimation_time_sec,
  stringsAsFactors = FALSE
)

benchmark_summary <- benchmark_summary[order(benchmark_summary$draws), ]
row.names(benchmark_summary) <- NULL

time_columns <- c(
  "jit_time_sec", "likelihood_eval_time_sec",
  "mean_estimation_time_sec", "postestimation_time_sec"
)
benchmark_summary[time_columns] <- lapply(
  benchmark_summary[time_columns],
  round,
  digits = 4
)
benchmark_summary[c("initial_loglik", "final_loglik")] <- lapply(
  benchmark_summary[c("initial_loglik", "final_loglik")],
  round,
  digits = 2
)

if (!requireNamespace("writexl", quietly = TRUE)) {
  install.packages("writexl")
}
writexl::write_xlsx(
  list(benchmark_summary = benchmark_summary),
  path = summary_file
)

print(benchmark_summary)
cat("Excel file saved to:", summary_file, "\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



  package implementation backend   device optimiser draws seed jit_time_sec
1    ANVL   anvl_bgw_gpu     GPU Tesla T4       BGW   500  123        1.788
2    ANVL   anvl_bgw_gpu     GPU Tesla T4       BGW  1000  123        1.588
3    ANVL   anvl_bgw_gpu     GPU Tesla T4       BGW  2000  123        1.610
4    ANVL   anvl_bgw_gpu     GPU Tesla T4       BGW  5000  123        1.639
5    ANVL   anvl_bgw_gpu     GPU Tesla T4       BGW 10000  123        1.830
  likelihood_eval_time_sec initial_loglik final_loglik mean_estimation_time_sec
1                    0.006       -2253.78     -1445.69                    0.367
2                    0.012       -2253.78     -1445.08                    0.499
3                    0.018       -2253.78     -1444.09                    0.655
4                    0.044       -2253.78     -1444.67                    1.266
5                    0.081       -2253.78     -1444.56                    2.205
  postestimation_time_sec
1                   0.333
2           